In [1]:
import pandas as pd
import re
import nltk
import joblib
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

nltk.download('punkt')
nltk.download('stopwords')

# Load data
df = pd.read_csv('../SMS/spam.csv', encoding='latin-1')[['v1','v2']]
df.columns = ['label', 'text']
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Preprocess
ps = PorterStemmer()
def clean(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z0-9]', ' ', text)
    text = re.sub(r'http\S+', '', text)
    tokens = nltk.word_tokenize(text)
    tokens = [ps.stem(w) for w in tokens if w.isalnum() and w not in stopwords.words('english')]
    return ' '.join(tokens)

df['clean'] = df['text'].apply(clean)

# Load vectorizer cũ
tfidf = joblib.load('vectorizer_sms.pkl')
X = tfidf.transform(df['clean']).toarray()
y = df['label_num'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train và save 4 models
joblib.dump(RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train), 'model_random_forest.pkl')
joblib.dump(DecisionTreeClassifier(random_state=42).fit(X_train, y_train),                   'model_decision_tree.pkl')
joblib.dump(MultinomialNB().fit(X_train, y_train),                                            'model_naive_bayes.pkl')
joblib.dump(LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train),         'model_logistic_regression.pkl')

print("Done! 4 models saved.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.5.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.5.2 when using version 1.7.2. This might lead to breaking code or invalid resu

Done! 4 models saved.
